In [4]:
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
import os

load_dotenv()

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_API_ENDPOINT = os.getenv("AZURE_OPENAI_API_ENDPOINT")
AZURE_OPENAI_API_DEPLOYMENT_4O = os.getenv("AZURE_OPENAI_API_DEPLOYMENT_4O")

llm_openai = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_API_ENDPOINT,
    deployment_name=AZURE_OPENAI_API_DEPLOYMENT_4O,
    openai_api_version=AZURE_OPENAI_API_VERSION,
    openai_api_key=AZURE_OPENAI_API_KEY
)





# Guiding in Prompts

In [5]:
result = llm_openai.invoke("Tell me a joke. Generate the output in key-value pair format with the following keys: setup, punchline.")
result.content

'setup: Why did the scarecrow win an award?  \npunchline: Because he was outstanding in his field!'

# Using Pydantic Models

In [6]:
from pydantic import BaseModel, Field

class llm_schema(BaseModel):
    setup: str = Field(description="The setup for the joke")
    punchline: str = Field(description="The punchline for the joke")


In [8]:
llm_structured_output = llm_openai.with_structured_output(llm_schema)

result = llm_structured_output.invoke("Tell me a joke")
result.punchline

'Because he was outstanding in his field!'

# Practice


In [21]:
from langchain_openai import AzureChatOpenAI
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import os
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# Initialize environment variables and Azure OpenAI client
# ============================================================

load_dotenv()

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_API_ENDPOINT = os.getenv("AZURE_OPENAI_API_ENDPOINT")
AZURE_OPENAI_API_DEPLOYMENT_4O = os.getenv("AZURE_OPENAI_API_DEPLOYMENT_4O")

llm_openai = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_API_ENDPOINT,
    deployment_name=AZURE_OPENAI_API_DEPLOYMENT_4O,
    openai_api_version=AZURE_OPENAI_API_VERSION,
    openai_api_key=AZURE_OPENAI_API_KEY
)

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    genre: str = Field(description="The genre of the movie")
    rating: float = Field(description="Rating from 0 to 10")

movie_llm = llm_openai.with_structured_output(Movie)

# Return a Json with:
# - title: The title of the movie
# - genre: The genre of the movie
# - rating: Rating from 0 to 10 

movie_result = movie_llm.invoke("Recommend me a movie")

print("\n=== Recommended Movie ===")
print(movie_result)
print("Title:", movie_result.title)
print("Rating type:", type(movie_result.rating))
print("Genre:", movie_result.genre)

# Order schema example
class Order(BaseModel):
    product_name: str = Field(description="The name of the product")
    quantity: int = Field(description="The quantity of the product")
    address: str = Field(description="The shipping address for the order")

order_llm = llm_openai.with_structured_output(Order)

order_input = "I want to buy 3 iPhones and ship to Taipei Main Station"

order_result = order_llm.invoke(order_input)

print("\n=== Order Details ===")
print(order_result)
print("Product Name:", order_result.product_name)
print("Quantity:", order_result.quantity)
print("Address:", order_result.address)

# Intention schema example
class Intent(BaseModel):
    intent: str = Field(description="refund | product_question | complaint")

intent_llm = llm_openai.with_structured_output(Intent)

intent_input = "This product is broken, I want my money back"
intent_result = intent_llm.invoke(intent_input)

print("\n=== Detected Intent ===")
print(intent_result)
print("Intent:", intent_result.intent)

# RAG
class Answer(BaseModel):
    answer: str = Field(description="The answer to the question based on the retrieved documents")
    source: str = Field(description="The source of the information used to answer the question")
    confidence: float = Field(description="The confidence score of the answer from 0 to 1")

answer_llm = llm_openai.with_structured_output(Answer)

answer_result = answer_llm.invoke("Explain what is LangChain")

print("\n=== RAG Answer ===")
print(answer_result)
print("Answer:", answer_result.answer)
print("Source:", answer_result.source)
print("Confidence:", answer_result.confidence)

class WeatherArgs(BaseModel):
    localtion: str = Field(description="City name")

# Tool Calling
class ToolCall(BaseModel):
    tool_name: str = Field(description="The name of the tool to call")
    arguments: WeatherArgs

tool_llm = llm_openai.with_structured_output(ToolCall)

tool_input = "What's the weather in Tokyo?"
tool_result = tool_llm.invoke(tool_input)

print("\n=== Tool Call ===")
print(tool_result)
print("Tool Name:", tool_result.tool_name)
print("Arguments:", tool_result.arguments)


=== Recommended Movie ===
title='Knives Out' genre='Mystery/Comedy' rating=8.1
Title: Knives Out
Rating type: <class 'float'>
Genre: Mystery/Comedy

=== Order Details ===
product_name='iPhone' quantity=3 address='Taipei Main Station'
Product Name: iPhone
Quantity: 3
Address: Taipei Main Station

=== Detected Intent ===
intent='refund'
Intent: refund

=== RAG Answer ===
answer='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It provides tools and abstractions that make it easy to build complex workflows involving LLMs, such as question answering, summarization, chatbots, and more. LangChain helps developers connect LLMs with various data sources, APIs, and enables features like memory (for context retention) and reasoning chains (multi-step logic). It is widely used for creating advanced AI applications that combine LLMs with external data and services.' source='https://www.langchain.com/how-it-works' confidence=0.95
A

In [ ]:
T